# TRIBE v2 — Backend
**Run Cell 1, then manually restart the kernel/runtime, then run Cells 2–10.**

In [ ]:
# ── CELL 1: Fix numpy (run this, then RESTART KERNEL before continuing) ───
!pip install -q numpy==2.2.6 --force-reinstall
print('\n✓ numpy==2.2.6 installed')
print('>>> NOW RESTART THE KERNEL/RUNTIME, then run from Cell 2 <<<')
print('    VS Code:  Kernel menu → Restart Kernel')
print('    Colab:    Runtime menu → Restart runtime')

In [ ]:
# ── CELL 2: Install all deps ───────────────────────────────────────────────
!apt-get install -y -q ffmpeg 2>/dev/null || echo 'ffmpeg: skipped (not apt env)'

import subprocess, sys
r = subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__)'],
                   capture_output=True, text=True)
torch_ver = r.stdout.strip()
if torch_ver:
    print(f'torch already installed: {torch_ver}')
else:
    print('Installing torch...')
    !pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

!pip install -q whisperx fastapi 'uvicorn[standard]' yt-dlp pydantic nest_asyncio huggingface_hub

# Re-pin numpy in case whisperx pulled it down
!pip install -q numpy==2.2.6 --force-reinstall

import numpy as np
import torch
print(f'numpy: {np.__version__}')
print(f'torch: {torch.__version__}')
assert np.__version__ == '2.2.6', f'STOP: numpy is {np.__version__}, need 2.2.6 — restart kernel and re-run Cell 1'

In [ ]:
# ── CELL 3: Clone + install tribev2 ───────────────────────────────────────
import os

clone_dir = '/content/tribev2' if os.path.exists('/content') else './tribev2'

if not os.path.exists(clone_dir):
    !git clone -q https://github.com/facebookresearch/tribev2 {clone_dir}
else:
    !git -C {clone_dir} pull -q
    print('tribev2 already present, pulled latest')

# --no-deps prevents pip from overwriting numpy/torch
!pip install -q -e {clone_dir} --no-deps
!pip install -q einops omegaconf

import tribev2
print(f'tribev2 OK: {tribev2.__file__}')

In [ ]:
# ── CELL 4: Inspect tribev2 — find real class names ───────────────────────
import tribev2, os, inspect, pkgutil, importlib

pkg_dir = os.path.dirname(os.path.abspath(tribev2.__file__))

print('=== .py files ===')
for root, dirs, files in os.walk(pkg_dir):
    dirs[:] = [d for d in dirs if d != '__pycache__']
    for f in files:
        if f.endswith('.py'):
            print(' ', os.path.relpath(os.path.join(root, f), pkg_dir))

print('\n=== Classes + public methods ===')
for _, modname, _ in pkgutil.walk_packages(tribev2.__path__, prefix='tribev2.'):
    try:
        mod = importlib.import_module(modname)
        for cname, cls in inspect.getmembers(mod, inspect.isclass):
            if (cls.__module__ or '').startswith('tribev2'):
                methods = [n for n, _ in inspect.getmembers(cls, callable) if not n.startswith('__')]
                print(f'  [{modname}] {cname}: {methods[:20]}')
    except Exception as e:
        print(f'  {modname}: skip ({e})')

print('\n=== __init__.py ===')
init = os.path.join(pkg_dir, '__init__.py')
print(open(init).read() if os.path.exists(init) else '(no __init__.py)')

In [ ]:
# ── CELL 5: Clone/update brain-neuro ──────────────────────────────────────
import os

repo_dir = '/content/brain-neuro' if os.path.exists('/content') else './brain-neuro'

if os.path.exists(repo_dir):
    !git -C {repo_dir} pull -q
    print('brain-neuro updated')
else:
    !git clone -q https://github.com/Sambhram1/brain-neuro.git {repo_dir}
    print('brain-neuro cloned')

os.chdir(repo_dir)
print(f'cwd: {os.getcwd()}')

In [ ]:
# ── CELL 6: Download model weights from HuggingFace ───────────────────────
import huggingface_hub

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    print('Using HF_TOKEN from Colab secrets')
except Exception:
    hf_token = input('Enter your HuggingFace token: ')

huggingface_hub.login(token=hf_token, add_to_git_credential=False)
huggingface_hub.hf_hub_download(
    repo_id='facebook/tribev2',
    filename='best.ckpt',
    local_dir='model_weights'
)
print('Weights downloaded.')

In [ ]:
# ── CELL 7: Patch config device ───────────────────────────────────────────
import re, torch

with open('model_weights/config.yaml') as f:
    cfg = f.read()

target = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg = re.sub(r'device: (cpu|cuda)', f'device: {target}', cfg)

with open('model_weights/config.yaml', 'w') as f:
    f.write(cfg)

print(f'Config patched → device: {target}')

In [ ]:
# ── CELL 8 (Optional): Upload cookies.txt for Instagram/TikTok ────────────
# Skip this cell if you only need YouTube.
import shutil

try:
    from google.colab import files
    print('Upload cookies.txt or skip')
    uploaded = files.upload()
    for fname in uploaded:
        shutil.move(fname, 'cookies.txt')
        print('cookies.txt saved')
except ImportError:
    # VS Code / local: place cookies.txt manually next to backend.py
    import os
    if os.path.exists('cookies.txt'):
        print('cookies.txt found')
    else:
        print('No cookies.txt — Instagram/TikTok may fail. Place it next to backend.py if needed.')

In [ ]:
# ── CELL 9: Install cloudflared tunnel ────────────────────────────────────
import platform, subprocess, os

if platform.system() == 'Linux':
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared
    print('cloudflared ready')
else:
    print('Non-Linux: install cloudflared manually from https://developers.cloudflare.com/cloudflared/')
    print('Then run: cloudflared tunnel --url http://localhost:8000')

In [ ]:
# ── CELL 10: Start backend + tunnel ───────────────────────────────────────
import nest_asyncio, uvicorn, threading, subprocess, time, re as _re, os

os.chdir('/content/brain-neuro' if os.path.exists('/content/brain-neuro') else './brain-neuro')
nest_asyncio.apply()

def run_server():
    uvicorn.run('backend:app', host='0.0.0.0', port=8000, log_level='warning')

threading.Thread(target=run_server, daemon=True).start()
time.sleep(4)
print('FastAPI started on port 8000')

proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print('Waiting for tunnel...')
for line in proc.stdout:
    match = _re.search(r'https://[\w-]+\.trycloudflare\.com', line.decode())
    if match:
        url = match.group()
        print(f'\n=======================================')
        print(f'  BACKEND URL: {url}')
        print(f'=======================================')
        print('Paste this into the frontend Backend field.')
        break